### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="infrared_thermography_temperature",
    dataset_year="2023",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    # https://physionet.org/content/face-oral-temp-data/1.0.0/
    original_dataset_source_download_link="https://doi.org/10.13026/9ay4-2c37",
    download_description="""
mkdir -p local-data-warehouse/infrared_thermography_temperature/ && wget https://archive.ics.uci.edu/static/public/925/data.csv -O local-data-warehouse/infrared_thermography_temperature/data.csv
""",
    # References
    academic_reference_bibtex="""@article{wang2023facial,
  title={Facial and oral temperature data from a large set of human subject volunteers},
  author={Wang, Quanzeng and Zhou, Yangling and Ghassemi, Pejman and Chenna, Dwith and Chen, Michelle and Casamento, Jon and Pfefer, Joshua and Mcbride, David},
  journal={PhysioNet, May},
  year={2023}
}
""",
    academic_reference_bibtex_key="wang2023facial",
    license="Creative Commons Zero 1.0 Universal Public Domain Dedication",
    data_tags=["IID"],
    curation_comments="""
- We use the temperature measured in monitor mode (aveOralM) as target because fast mode (aveOralF) is stated to be less accurate.
- We exclude T_atm, Humidity, and Distance since they are control parameters from the lab scenario and would not be available in a real-world deployment scenario.
- We exclude T_offset1, because it is a derived feature that is not available in a real-world deployment scenario.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="aveOralM",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "data.csv")
df = df.drop(columns=["aveOralF", "T_atm", "Humidity", "Distance", "T_offset1"])
print("Loaded data shape:", df.shape)

In [ ]:
df["SubjectID"].value_counts()

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)